## NumPy

### 1.1 Why NumPy?

**NumPy (Numerical Python)** is the foundation of the scientific Python ecosystem. Pandas, scikit-learn, SciPy, and even deep-learning frameworks are built on top of (or inspired by) NumPy.

**Why not just use Python lists?**

| Feature | Python `list` | NumPy `ndarray` |
|---|---|---|
| Element types | Mixed (anything) | Homogeneous (one dtype) |
| Memory layout | Pointers scattered in memory | Contiguous block of memory |
| Speed | Slow loops in pure Python | Vectorized C loops (10–100× faster) |
| Math operations | Not element-wise (`[1,2]*2 → [1,2,1,2]`) | Element-wise (`[1,2]*2 → [2,4]`) |
| Broadcasting | Not available | Built-in |

Let's prove the speed claim first.

In [2]:
import numpy as np
import pandas as pd

print("NumPy version :", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version : 2.4.6
Pandas version: 3.0.3


In [3]:
# Speed comparison: squaring 1 million numbers
big_list = list(range(1_000_000))
big_array = np.arange(1_000_000)

# Pure Python
%timeit [x ** 2 for x in big_list]

# NumPy (vectorized)
%timeit big_array ** 2

73.9 ms ± 8.84 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
2.52 ms ± 97.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


NumPy is dramatically faster because the loop runs in compiled C code, operating on a contiguous block of memory — not on a million separate Python objects.

### 1.2 Creating Arrays

There are many ways to create an `ndarray`:

| Function | What it creates |
|---|---|
| `np.array(seq)` | Array from a list / tuple / nested list |
| `np.zeros(shape)` | All zeros |
| `np.ones(shape)` | All ones |
| `np.full(shape, v)` | All filled with value `v` |
| `np.arange(start, stop, step)` | Like `range()`, but returns an array |
| `np.linspace(a, b, n)` | `n` evenly spaced points from `a` to `b` (inclusive) |
| `np.eye(n)` | Identity matrix |
| `np.random.*` | Random numbers |

In [4]:
# From Python lists
a = np.array([1, 2, 3, 4])                 # 1-D array (vector)
b = np.array([[1, 2, 3], [4, 5, 6]])       # 2-D array (matrix) from nested lists

print("a =", a)
print("b =\n", b)

a = [1 2 3 4]
b =
 [[1 2 3]
 [4 5 6]]


In [5]:
print("zeros:\n", np.zeros((2, 3)))            # 2 rows, 3 cols of 0.0
print("\nones:\n", np.ones((2, 2)))
print("\nfull:\n", np.full((2, 3), 7))
print("\narange:", np.arange(0, 10, 2))         # start, stop (exclusive), step
print("linspace:", np.linspace(0, 1, 5))        # 5 points from 0 to 1, inclusive
print("\nidentity:\n", np.eye(3))

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]]

ones:
 [[1. 1.]
 [1. 1.]]

full:
 [[7 7 7]
 [7 7 7]]

arange: [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]

identity:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [5]:
# Random arrays — use a Generator with a seed for reproducible results
rng = np.random.default_rng(seed=42)

print("uniform [0,1):\n", rng.random((2, 3)))
print("\nintegers 1..9:\n", rng.integers(1, 10, size=(3, 3)))
print("\nstandard normal:\n", rng.normal(loc=0, scale=1, size=5))

uniform [0,1):
 [[0.77395605 0.43887844 0.85859792]
 [0.69736803 0.09417735 0.97562235]]

integers 1..9:
 [[7 7 7]
 [8 5 2]
 [8 5 5]]

standard normal:
 [ 0.77779194  0.0660307   1.12724121  0.46750934 -0.85929246]


### 1.3 Array Attributes & Data Types (`dtype`)

Every array has:

- **`ndim`** — number of dimensions (axes)
- **`shape`** — tuple with the size along each axis
- **`size`** — total number of elements
- **`dtype`** — the data type of *every* element (arrays are homogeneous)
- **`itemsize`** / **`nbytes`** — bytes per element / total bytes

The **dtype** is crucial: it determines memory usage, precision, and what operations are valid. Common dtypes: `int8/16/32/64`, `uint8...`, `float32/64`, `bool`, `complex128`, and `<U...` for fixed-width Unicode strings.

In [6]:
M = np.array([[1, 2, 3], [4, 5, 6]])

print("ndim    :", M.ndim)        # 2 axes → axis 0 = rows, axis 1 = columns
print("shape   :", M.shape)       # (2, 3)
print("size    :", M.size)        # 6 elements
print("dtype   :", M.dtype)       # platform default integer (usually int64)
print("itemsize:", M.itemsize, "bytes per element")
print("nbytes  :", M.nbytes, "bytes total")

ndim    : 2
shape   : (2, 3)
size    : 6
dtype   : int64
itemsize: 8 bytes per element
nbytes  : 48 bytes total


In [7]:
# Controlling and converting dtypes
ints   = np.array([1, 2, 3], dtype=np.int32)
floats = np.array([1, 2, 3], dtype=np.float64)
mixed  = np.array([1, 2.5, 3])        # int + float → upcast to float64

print(ints.dtype, floats.dtype, mixed.dtype)

# astype() returns a NEW array with the converted type
f = np.array([1.7, 2.2, 3.9])
print("float → int (truncates, not rounds):", f.astype(np.int64))
print("int → bool (0 is False):", np.array([0, 1, 2, 0]).astype(bool))

int32 float64 float64
float → int (truncates, not rounds): [1 2 3]
int → bool (0 is False): [False  True  True False]


> ⚠️ **Watch out:** converting float → int *truncates* toward zero (1.9 → 1). Use `np.round()` first if you want rounding. Also, integer dtypes can **overflow** silently — e.g. `np.int8` only holds −128…127.

### 1.4 Indexing & Slicing

1-D arrays index just like Python lists. For multi-dimensional arrays, use a **comma-separated tuple**: `arr[row, col]`.

The golden rule for slices: `arr[start:stop:step]` — `stop` is exclusive.

In [8]:
x = np.arange(10, 20)     # [10 11 12 13 14 15 16 17 18 19]
print("x        :", x)
print("x[0]     :", x[0])
print("x[-1]    :", x[-1])      # last element
print("x[2:5]   :", x[2:5])     # indices 2,3,4
print("x[::2]   :", x[::2])     # every 2nd element
print("x[::-1]  :", x[::-1])    # reversed

x        : [10 11 12 13 14 15 16 17 18 19]
x[0]     : 10
x[-1]    : 19
x[2:5]   : [12 13 14]
x[::2]   : [10 12 14 16 18]
x[::-1]  : [19 18 17 16 15 14 13 12 11 10]


In [3]:
# 2-D indexing
M = np.arange(1, 13).reshape(3, 4)   # 3 rows × 4 cols
print("M =\n", M)

print("\nM[1, 2]  :", M[1, 2])      # row 1, col 2 → single element
print("M[0]     :", M[0])            # entire first row
print("M[:, 1]  :", M[:, 1])         # entire 2nd column  (':' = all rows)
print("M[0:2, 1:3]:\n", M[0:2, 1:3])# sub-matrix: rows 0-1, cols 1-2
print("M[-1, :] :", M[-1, :])        # last row

M =
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

M[1, 2]  : 7
M[0]     : [1 2 3 4]
M[:, 1]  : [ 2  6 10]
M[0:2, 1:3]:
 [[2 3]
 [6 7]]
M[-1, :] : [ 9 10 11 12]


### 1.6 Vectorized Operations (ufuncs)

Arithmetic on arrays happens **element-wise** with no explicit loop. These optimized functions are called **universal functions (ufuncs)**: `+ - * / ** %`, `np.sqrt`, `np.exp`, `np.log`, `np.sin`, comparisons, etc.

In [10]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print("a + b  :", a + b)
print("a * b  :", a * b)          # element-wise, NOT matrix multiplication
print("b / a  :", b / a)
print("a ** 2 :", a ** 2)
print("sqrt(b):", np.sqrt(b))
print("a @ b  :", a @ b)          # dot product (matrix multiply for 2-D)

a + b  : [11 22 33 44]
a * b  : [ 10  40  90 160]
b / a  : [10. 10. 10. 10.]
a ** 2 : [ 1  4  9 16]
sqrt(b): [3.16227766 4.47213595 5.47722558 6.32455532]
a @ b  : 300


### 1.7 Broadcasting ⭐

**Broadcasting** is the set of rules NumPy uses to perform operations on arrays of *different shapes* — without making physical copies of data.

**The rules** (compare shapes from the *rightmost* dimension, moving left):

1. If the dimensions are **equal** → compatible.
2. If one of them is **1** → it is *stretched* to match the other.
3. If a shape is **missing dimensions** → it is padded with 1s on the left.
4. Otherwise → `ValueError: operands could not be broadcast together`.

```
A: (3, 4)        A: (3, 4)        A: (3, 1)        A: (3, 2)
B:    (4,) ✔     B: (3, 1) ✔      B:    (4,) ✔     B:    (4,) ✘
→  (3, 4)        →  (3, 4)        →  (3, 4)        incompatible!
```

Example 1: Scalar Broadcasting

In [12]:
import numpy as np

a = np.array([1, 2, 3])
b = 10

print(a + b)

[11 12 13]


Here, NumPy treats 10 as if it were [10, 10, 10].\
Example 2: Array Broadcasting

In [14]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])

b = np.array([10, 20, 30])

print(a + b)

[[11 22 33]
 [14 25 36]]


Example 3: Column Vector Broadcasting

In [16]:
a = np.array([[1],
              [2],
              [3]])      # shape (3,1)

b = np.array([10, 20, 30])  # shape (3,)
print(a + b)

[[11 21 31]
 [12 22 32]
 [13 23 33]]


Example of Incompatible Shapes

In [15]:
a = np.array([[1, 2],
              [3, 4]])   # shape (2,2)

b = np.array([10, 20, 30])  # shape (3,)

a + b

ValueError: operands could not be broadcast together with shapes (2,2) (3,) 

### 1.8 Aggregations & the `axis` Argument

Aggregation functions reduce an array to fewer values: `sum, mean, std, var, min, max, argmin, argmax, cumsum, ...`

The **`axis`** parameter tells NumPy *which dimension to collapse*:

- `axis=0` → collapse rows → one result **per column**
- `axis=1` → collapse columns → one result **per row**
- no `axis` → aggregate over the whole array

In [17]:
M = np.array([[4, 9, 2],
              [8, 1, 6],
              [3, 5, 7]])

print("total sum        :", M.sum())
print("sum per column   :", M.sum(axis=0))
print("sum per row      :", M.sum(axis=1))
print("max per column   :", M.max(axis=0))
print("index of row max :", M.argmax(axis=1))   # position of max within each row
print("mean / std       :", M.mean().round(2), "/", M.std().round(2))

total sum        : 45
sum per column   : [15 15 15]
sum per row      : [15 15 15]
max per column   : [8 9 7]
index of row max : [1 0 2]
mean / std       : 5.0 / 2.58
